In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/customer_support.csv")
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [2]:
df.shape

(26872, 5)

In [3]:
df.columns

Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='str')

In [4]:
df["intent"].nunique()

27

In [5]:
df["category"].value_counts()

category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

In [6]:
df["intent"].value_counts()

intent
check_invoice               1000
complaint                   1000
contact_customer_service    1000
edit_account                1000
switch_account              1000
check_payment_methods        999
contact_human_agent          999
delivery_period              999
get_invoice                  999
newsletter_subscription      999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
set_up_shipping_address      997
delete_account               995
delivery_options             995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64

In [7]:
df.isna().sum()

flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

In [8]:
df[["instruction", "intent"]].sample(10, random_state=42)

,instruction,intent
9329,I can't talk with a human agent,contact_human_agent
4160,I have got to locate hte bills from {{Person N...,check_invoice
18500,"I cannot pay, help me to inform of a problem w...",payment_issue
8840,I want help speaking to customer service,contact_customer_service
5098,I try to see th accepted payment options,check_payment_methods
17250,where to sign up to the company nmewsletter,newsletter_subscription
3589,I'd like to see the withdrwaal fee how can i d...,check_cancellation_fee
9043,I want to speak with someone,contact_human_agent
15800,can you help me getting bill #85632?,get_invoice
4384,I don't know how to take a quick look at invoi...,check_invoice


In [9]:
X = df["instruction"]
y = df["intent"]

In [10]:
print(X.shape)
print(y.shape)

(26872,)
(26872,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
print(X_train.shape)
print(X_test.shape)

(21497,)
(5375,)


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline_logistic = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=2000
        )
    )
])

In [14]:
pipeline_logistic.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](27,)","['cancel_order','change_order','change_shipping_address',..., 'switch_account','track_order','track_refund']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [15]:
previsoes = pipeline_logistic.predict(X_test)

In [16]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_test, previsoes)
macro_f1 = f1_score(y_test, previsoes, average="macro")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)

Accuracy: 0.994046511627907
Macro F1: 0.994058929394015


In [17]:
from sklearn.metrics import classification_report

print(classification_report(y_test, previsoes))

                          precision    recall  f1-score   support

            cancel_order       1.00      0.98      0.99       200
            change_order       0.97      0.98      0.97       199
 change_shipping_address       0.99      1.00      1.00       195
  check_cancellation_fee       0.99      1.00      1.00       190
           check_invoice       0.97      1.00      0.99       200
   check_payment_methods       1.00      0.99      1.00       200
     check_refund_policy       1.00      0.99      1.00       199
               complaint       1.00      1.00      1.00       200
contact_customer_service       1.00      0.99      0.99       200
     contact_human_agent       0.99      0.99      0.99       200
          create_account       0.99      0.99      0.99       199
          delete_account       0.99      1.00      0.99       199
        delivery_options       1.00      1.00      1.00       199
         delivery_period       1.00      1.00      1.00       200
         

In [18]:
erros = pd.DataFrame({
    "texto": X_test,
    "real": y_test,
    "previsto": previsoes
})

erros = erros[erros["real"] != erros["previsto"]]

erros.head(20)

,texto,real,previsto
15879,I do not know how I candownload my bill #37777,get_invoice,check_invoice
21728,help creating account,registration_problems,create_account
6860,am I entitled to a reimbursement?,check_refund_policy,get_refund
8438,I don't know what to do to speak with cusstome...,contact_customer_service,contact_human_agent
18023,need assistance reporting an issue with pyments,payment_issue,registration_problems
17941,where can I notify of problems with payent?,payment_issue,registration_problems
10501,I need a user,create_account,edit_account
10358,what do i have to do to createa premium account,create_account,delete_account
1523,can I update order00004587345?,change_order,change_shipping_address
15091,i cannot find bill #85632 can usendit to me,get_invoice,check_invoice


In [19]:
len(erros)

32

In [20]:
erros.head(20)

,texto,real,previsto
15879,I do not know how I candownload my bill #37777,get_invoice,check_invoice
21728,help creating account,registration_problems,create_account
6860,am I entitled to a reimbursement?,check_refund_policy,get_refund
8438,I don't know what to do to speak with cusstome...,contact_customer_service,contact_human_agent
18023,need assistance reporting an issue with pyments,payment_issue,registration_problems
17941,where can I notify of problems with payent?,payment_issue,registration_problems
10501,I need a user,create_account,edit_account
10358,what do i have to do to createa premium account,create_account,delete_account
1523,can I update order00004587345?,change_order,change_shipping_address
15091,i cannot find bill #85632 can usendit to me,get_invoice,check_invoice


In [21]:
erros[["real", "previsto"]].value_counts()

real                      previsto               
get_invoice               check_invoice              5
track_order               change_order               4
payment_issue             registration_problems      3
change_order              edit_account               3
registration_problems     create_account             2
contact_customer_service  contact_human_agent        2
cancel_order              change_order               2
check_refund_policy       get_refund                 1
create_account            edit_account               1
                          delete_account             1
change_order              change_shipping_address    1
check_payment_methods     check_cancellation_fee     1
switch_account            delete_account             1
contact_human_agent       get_refund                 1
cancel_order              delete_account             1
track_order               check_invoice              1
place_order               change_order               1
track_order    

In [22]:
from sklearn.svm import LinearSVC

pipeline_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "modelo",
        LinearSVC()
    )
])

In [23]:
pipeline_svm.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](27,)","['cancel_order','change_order','change_shipping_address',..., 'switch_account','track_order','track_refund']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [24]:
accuracy_svm = accuracy_score(y_test, previsoes_svm)
macro_f1_svm = f1_score(y_test, previsoes_svm, average="macro")

print("Accuracy SVM:", accuracy_svm)
print("Macro F1 SVM:", macro_f1_svm)

NameError: name 'previsoes_svm' is not defined

In [25]:
previsoes_svm = pipeline_svm.predict(X_test)

In [26]:
accuracy_svm = accuracy_score(y_test, previsoes_svm)
macro_f1_svm = f1_score(y_test, previsoes_svm, average="macro")

print("Accuracy SVM:", accuracy_svm)
print("Macro F1 SVM:", macro_f1_svm)

Accuracy SVM: 0.9951627906976744
Macro F1 SVM: 0.9951728838388713


In [27]:
erros_svm = pd.DataFrame({
    "texto": X_test,
    "real": y_test,
    "previsto": previsoes_svm
})

erros_svm = erros_svm[erros_svm["real"] != erros_svm["previsto"]]

len(erros_svm)

26

In [28]:
erros_svm[["real", "previsto"]].value_counts()

real                      previsto               
track_order               change_order               4
get_invoice               check_invoice              3
change_order              edit_account               3
payment_issue             registration_problems      2
cancel_order              change_order               2
delivery_period           delivery_options           1
registration_problems     create_account             1
                          payment_issue              1
check_refund_policy       get_refund                 1
create_account            edit_account               1
                          delete_account             1
change_order              change_shipping_address    1
switch_account            delete_account             1
cancel_order              delete_account             1
check_invoice             get_invoice                1
contact_customer_service  contact_human_agent        1
set_up_shipping_address   change_shipping_address    1
Name: count, dt

In [29]:
erros_svm.head(20)

,texto,real,previsto
13067,I am trying to find the shipping perod,delivery_period,delivery_options
15879,I do not know how I candownload my bill #37777,get_invoice,check_invoice
21728,help creating account,registration_problems,create_account
21577,need help informing of rergistration errors,registration_problems,payment_issue
6860,am I entitled to a reimbursement?,check_refund_policy,get_refund
18023,need assistance reporting an issue with pyments,payment_issue,registration_problems
10501,I need a user,create_account,edit_account
10358,what do i have to do to createa premium account,create_account,delete_account
1523,can I update order00004587345?,change_order,change_shipping_address
15091,i cannot find bill #85632 can usendit to me,get_invoice,check_invoice


In [30]:
from sklearn.model_selection import cross_val_score

cv_logistic = cross_val_score(
    pipeline_logistic,
    X,
    y,
    cv=5,
    scoring="f1_macro"
)

print(cv_logistic)
print("Macro F1 médio Logistic:", cv_logistic.mean())
print("Desvio padrão:", cv_logistic.std())

[0.99294687 0.99256544 0.99293659 0.99257019 0.99164969]
Macro F1 médio Logistic: 0.9925337571364178
Desvio padrão: 0.0004726207162377721


In [31]:
cv_svm = cross_val_score(
    pipeline_svm,
    X,
    y,
    cv=5,
    scoring="f1_macro"
)

print(cv_svm)
print("Macro F1 médio SVM:", cv_svm.mean())
print("Desvio padrão:", cv_svm.std())

[0.99517392 0.99498055 0.99405742 0.99498161 0.99238508]
Macro F1 médio SVM: 0.9943157152755365
Desvio padrão: 0.0010407715023896223


In [32]:
cv_logistic = cross_val_score(
    pipeline_logistic,
    X,
    y,
    cv=5,
    scoring="f1_macro"
)

print(cv_logistic)
print("Macro F1 médio Logistic:", cv_logistic.mean())
print("Desvio padrão:", cv_logistic.std())

[0.99294687 0.99256544 0.99293659 0.99257019 0.99164969]
Macro F1 médio Logistic: 0.9925337571364178
Desvio padrão: 0.0004726207162377721


In [33]:
pipeline_final = pipeline_svm

pipeline_final.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](27,)","['cancel_order','change_order','change_shipping_address',..., 'switch_account','track_order','track_refund']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [34]:
import joblib

joblib.dump(
    pipeline_final,
    "../models/nlp_intent_classifier.joblib"
)

['../models/nlp_intent_classifier.joblib']

In [35]:
def prever_intent(texto):
    intent = pipeline_final.predict([texto])[0]
    return intent

In [36]:
prever_intent("I want to cancel my order")

'cancel_order'

In [37]:
print(prever_intent("Where is my package?"))
print(prever_intent("I forgot my password"))
print(prever_intent("I was charged but the payment failed"))
print(prever_intent("Can I get my money back?"))
print(prever_intent("I want to talk to a real person"))

delivery_period
recover_password
payment_issue
get_refund
contact_human_agent


In [38]:
intent_para_categoria = (
    df[["intent", "category"]]
    .drop_duplicates()
    .set_index("intent")["category"]
    .to_dict()
)

In [39]:
def prever_intent(texto):
    intent = pipeline_final.predict([texto])[0]
    categoria = intent_para_categoria[intent]

    return {
        "intent": intent,
        "category": categoria
    }

In [40]:
prever_intent("I forgot my password")

{'intent': 'recover_password', 'category': 'ACCOUNT'}

In [41]:
{
    "intent": "recover_password",
    "category": "ACCOUNT"
}

{'intent': 'recover_password', 'category': 'ACCOUNT'}

In [42]:
prever_intent("Where is my package?")

{'intent': 'delivery_period', 'category': 'DELIVERY'}

In [43]:
import numpy as np

def softmax(scores):
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / exp_scores.sum()

In [44]:
def prever_intent(texto):
    intent = pipeline_final.predict([texto])[0]
    categoria = intent_para_categoria[intent]

    scores = pipeline_final.decision_function([texto])[0]
    probabilidades = softmax(scores)

    classes = pipeline_final.classes_
    indice = list(classes).index(intent)

    confianca = float(probabilidades[indice])

    return {
        "intent": intent,
        "category": categoria,
        "confidence": round(confianca * 100, 1)
    }

In [45]:
prever_intent("I forgot my password")

{'intent': 'recover_password', 'category': 'ACCOUNT', 'confidence': 19.7}

In [46]:
prever_intent("Where is my package?")

{'intent': 'delivery_period', 'category': 'DELIVERY', 'confidence': 7.9}

In [47]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

pipeline_svm_calibrado = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "modelo",
        CalibratedClassifierCV(
            LinearSVC(),
            method="sigmoid",
            cv=5
        )
    )
])

In [48]:
pipeline_svm_calibrado.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](27,)","['cancel_order','change_order','change_shipping_address',..., 'switch_account','track_order','track_refund']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [49]:
previsoes_calibradas = pipeline_svm_calibrado.predict(X_test)

In [50]:
accuracy_calibrada = accuracy_score(y_test, previsoes_calibradas)
macro_f1_calibrado = f1_score(
    y_test,
    previsoes_calibradas,
    average="macro"
)

print("Accuracy:", accuracy_calibrada)
print("Macro F1:", macro_f1_calibrado)

Accuracy: 0.9947906976744186
Macro F1: 0.9948039244448018


In [51]:
pipeline_final = pipeline_svm_calibrado

pipeline_final.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](27,)","['cancel_order','change_order','change_shipping_address',..., 'switch_account','track_order','track_refund']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [52]:
joblib.dump(
    pipeline_final,
    "../models/nlp_intent_classifier.joblib"
)

['../models/nlp_intent_classifier.joblib']

In [53]:
joblib.dump(
    intent_para_categoria,
    "../models/intent_to_category.joblib"
)

['../models/intent_to_category.joblib']

In [54]:
def prever_intent(texto):
    probabilidades = pipeline_final.predict_proba([texto])[0]

    indice = probabilidades.argmax()
    intent = pipeline_final.classes_[indice]

    categoria = intent_para_categoria[intent]
    confianca = float(probabilidades[indice])

    return {
        "intent": intent,
        "category": categoria,
        "confidence": round(confianca * 100, 1)
    }

In [55]:
print(prever_intent("I forgot my password"))
print(prever_intent("Where is my package?"))
print(prever_intent("I want to speak to a real person"))

{'intent': 'recover_password', 'category': 'ACCOUNT', 'confidence': 98.7}
{'intent': 'delivery_period', 'category': 'DELIVERY', 'confidence': 62.6}
{'intent': 'contact_human_agent', 'category': 'CONTACT', 'confidence': 98.5}


In [56]:
testes = [
    "I need help",
    "My order has not arrived yet",
    "Can you change the address for my order?",
    "I want my money back",
    "I cannot create an account",
    "What payment methods do you accept?",
    "I want to cancel everything",
    "Can I speak to customer support?"
]

for texto in testes:
    print(texto)
    print(prever_intent(texto))
    print()

I need help
{'intent': 'get_refund', 'category': 'REFUND', 'confidence': 27.5}

My order has not arrived yet
{'intent': 'create_account', 'category': 'ACCOUNT', 'confidence': 50.6}

Can you change the address for my order?
{'intent': 'change_shipping_address', 'category': 'SHIPPING', 'confidence': 99.5}

I want my money back
{'intent': 'get_refund', 'category': 'REFUND', 'confidence': 98.7}

I cannot create an account
{'intent': 'create_account', 'category': 'ACCOUNT', 'confidence': 93.0}

What payment methods do you accept?
{'intent': 'check_payment_methods', 'category': 'PAYMENT', 'confidence': 99.0}

I want to cancel everything
{'intent': 'cancel_order', 'category': 'ORDER', 'confidence': 50.9}

Can I speak to customer support?
{'intent': 'contact_customer_service', 'category': 'CONTACT', 'confidence': 99.3}



In [57]:
def prever_top3(texto):
    probabilidades = pipeline_final.predict_proba([texto])[0]
    classes = pipeline_final.classes_

    indices = probabilidades.argsort()[-3:][::-1]

    resultados = []

    for indice in indices:
        intent = classes[indice]

        resultados.append({
            "intent": intent,
            "category": intent_para_categoria[intent],
            "confidence": round(float(probabilidades[indice]) * 100, 1)
        })

    return resultados

In [58]:
print(prever_top3("My order has not arrived yet"))
print(prever_top3("I need help"))
print(prever_top3("I want to cancel everything"))

[{'intent': 'create_account', 'category': 'ACCOUNT', 'confidence': 50.6}, {'intent': 'recover_password', 'category': 'ACCOUNT', 'confidence': 8.7}, {'intent': 'place_order', 'category': 'ORDER', 'confidence': 8.2}]
[{'intent': 'get_refund', 'category': 'REFUND', 'confidence': 27.5}, {'intent': 'place_order', 'category': 'ORDER', 'confidence': 25.4}, {'intent': 'cancel_order', 'category': 'ORDER', 'confidence': 19.4}]
[{'intent': 'cancel_order', 'category': 'ORDER', 'confidence': 50.9}, {'intent': 'delete_account', 'category': 'ACCOUNT', 'confidence': 48.1}, {'intent': 'newsletter_subscription', 'category': 'SUBSCRIPTION', 'confidence': 0.5}]


In [59]:
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [60]:
X_embeddings = modelo_embeddings.encode(
    df["instruction"].tolist(),
    show_progress_bar=True
)

X_embeddings.shape

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

(26872, 384)

In [61]:
X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_embeddings,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [62]:
modelo_semantico = LogisticRegression(
    max_iter=3000
)

modelo_semantico.fit(X_train_emb, y_train_emb)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [63]:
previsoes_semanticas = modelo_semantico.predict(X_test_emb)

accuracy_semantica = accuracy_score(
    y_test_emb,
    previsoes_semanticas
)

f1_semantico = f1_score(
    y_test_emb,
    previsoes_semanticas,
    average="macro"
)

print("Accuracy:", accuracy_semantica)
print("Macro F1:", f1_semantico)

Accuracy: 0.994046511627907
Macro F1: 0.9940457624937806


In [64]:
def prever_semantico(texto):
    embedding = modelo_embeddings.encode([texto])

    probabilidades = modelo_semantico.predict_proba(embedding)[0]
    indice = probabilidades.argmax()

    intent = modelo_semantico.classes_[indice]
    confianca = float(probabilidades[indice])

    return {
        "intent": intent,
        "category": intent_para_categoria[intent],
        "confidence": round(confianca * 100, 1)
    }

In [65]:
testes_semanticos = [
    "My order has not arrived yet",
    "I need help",
    "I want to cancel everything",
    "Where is my package?",
    "I forgot my password",
    "Can you change the address for my order?"
]

for texto in testes_semanticos:
    print(texto)
    print(prever_semantico(texto))
    print()

My order has not arrived yet
{'intent': 'delivery_period', 'category': 'DELIVERY', 'confidence': 63.7}

I need help
{'intent': 'place_order', 'category': 'ORDER', 'confidence': 34.1}

I want to cancel everything
{'intent': 'cancel_order', 'category': 'ORDER', 'confidence': 42.8}

Where is my package?
{'intent': 'delivery_options', 'category': 'DELIVERY', 'confidence': 39.3}

I forgot my password
{'intent': 'recover_password', 'category': 'ACCOUNT', 'confidence': 96.8}

Can you change the address for my order?
{'intent': 'change_shipping_address', 'category': 'SHIPPING', 'confidence': 91.0}



In [66]:
import joblib

joblib.dump(
    modelo_semantico,
    "../models/semantic_intent_classifier.joblib"
)

['../models/semantic_intent_classifier.joblib']

In [67]:
joblib.dump(
    intent_para_categoria,
    "../models/intent_to_category.joblib"
)

['../models/intent_to_category.joblib']

In [68]:
from sklearn.linear_model import LogisticRegression

modelo_semantico_final = LogisticRegression(
    max_iter=3000
)

modelo_semantico_final.fit(X_embeddings, y)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [69]:
import joblib

joblib.dump(
    modelo_semantico_final,
    "../models/semantic_intent_classifier.joblib"
)

joblib.dump(
    intent_para_categoria,
    "../models/intent_to_category.joblib"
)

['../models/intent_to_category.joblib']

In [70]:
def prever_semantico_final(texto):
    embedding = modelo_embeddings.encode([texto])

    probabilidades = modelo_semantico_final.predict_proba(embedding)[0]
    classes = modelo_semantico_final.classes_

    indices_top3 = probabilidades.argsort()[-3:][::-1]

    top3 = []

    for indice in indices_top3:
        intent = classes[indice]

        top3.append({
            "intent": intent,
            "category": intent_para_categoria[intent],
            "confidence": round(float(probabilidades[indice]) * 100, 1)
        })

    principal = top3[0]

    if principal["confidence"] >= 70:
        nivel = "High"
    elif principal["confidence"] >= 40:
        nivel = "Medium"
    else:
        nivel = "Low"

    return {
        "intent": principal["intent"],
        "category": principal["category"],
        "confidence": principal["confidence"],
        "confidence_level": nivel,
        "top_3": top3
    }

In [71]:
prever_semantico_final("My order has not arrived yet")

{'intent': 'delivery_period',
 'category': 'DELIVERY',
 'confidence': 65.7,
 'confidence_level': 'Medium',
 'top_3': [{'intent': 'delivery_period',
   'category': 'DELIVERY',
   'confidence': 65.7},
  {'intent': 'place_order', 'category': 'ORDER', 'confidence': 8.7},
  {'intent': 'track_order', 'category': 'ORDER', 'confidence': 6.1}]}